# Time Series Analysis of Stock Prices (Apple)
This notebook reproduces the analysis performed in `analysis.py`. It downloads historical price data, performs trend analysis, seasonality decomposition, and forecasts the next 180 days using Prophet.
The generated plots are saved as PNG files and compiled into a short demo video `demo_video.mp4`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from statsmodels.tsa.seasonal import seasonal_decompose
from prophet import Prophet

STOCK_TICKER = "AAPL"
START_DATE = "2015-01-01"
END_DATE = "2024-12-31"

df = yf.download(STOCK_TICKER, start=START_DATE, end=END_DATE)
if "Adj Close" in df.columns:
    ts = df["Adj Close"].dropna()
else:
    ts = df["Close"].dropna()
ts.index = pd.to_datetime(ts.index)

# Raw price plot
ts.plot(title=f"{STOCK_TICKER} Adjusted Close Price")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.tight_layout()
plt.savefig("price_plot.png")
plt.close()

# Trend analysis (30‑day rolling mean)
rolling_mean = ts.rolling(window=30, center=True).mean()
plt.figure(figsize=(10,5))
plt.plot(ts, label="Adj Close", alpha=0.6)
plt.plot(rolling_mean, label="30‑Day Rolling Mean", color="red")
plt.title("Trend Analysis")
plt.legend()
plt.savefig("trend_plot.png")
plt.close()

# Seasonality detection (weekly resample)
ts_weekly = ts.resample('W').mean()
result = seasonal_decompose(ts_weekly, model='additive', period=52)
result.plot()
plt.tight_layout()
plt.savefig("seasonality_decompose.png")
plt.close()

# Forecast with Prophet
prophet_df = ts.reset_index()
prophet_df.columns = ["ds", "y"]
model = Prophet(yearly_seasonality=True, daily_seasonality=False, weekly_seasonality=False)
model.fit(prophet_df)
future = model.make_future_dataframe(periods=180)
forecast = model.predict(future)
model.plot(forecast)
plt.title("Forecast (180 days)")
plt.savefig("forecast.png")
plt.close()
